# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/furkankumrudev/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
%pip -q install duckdb

In [8]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connection ready.")

DuckDB connection ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item for one client on one report date.

I will use daily search performance data over a 90-day window, using March 2026 as the development month.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {fact_daily}
""").df()

check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

- `gsc_impressions`: Measures search visibility for a content item.
- `gsc_clicks`: Measures search traffic generated by the content item.
- `gsc_avg_position`: Measures the average search ranking position.
- `report_date`: Used to define the time windows for the analysis.

### Label / proxy

- `is_declining`: A proxy target indicating whether a content item experiences a decline in search impressions.

### Context

- `client_hash_id`: Identifies the client associated with the content item. It is used for grouping and validation, not as a model feature.
- `content_hash_id`: Identifies the content item and is used for grouping and joins, not as a model feature.
- `report_date`: Provides the time context for the observations.

### Excluded

- I deliberately exclude `ga4_*` fields because this lane focuses on search performance, while GA4 availability differs across clients and time periods.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the fields used in the contract
field_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(gsc_impressions) AS impressions_available,
    COUNT(gsc_clicks) AS clicks_available,
    COUNT(gsc_avg_position) AS position_available,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

print("Field availability and context check for March 2026:")
display(field_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Field availability and context check for March 2026:


,total_rows,impressions_available,clicks_available,position_available,clients,content_items
0,9841378,9841378,9841378,3611061,55,331437


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Grain check
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM {fact_daily}
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate grain rows:")
display(grain_check)


# 2. Row count + date span
count_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {fact_daily}
""").df()

print("March 2026 row count and date span:")
display(count_check)


# 3. Availability check
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) * 100.0 / COUNT(*) AS available_pct
FROM {fact_daily}
""").df()

print("GSC availability:")
display(availability_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows:


,report_date,client_hash_id,content_hash_id,n


March 2026 row count and date span:


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GSC availability:


,total_rows,available_rows,available_pct
0,9841378,3611061,36.692636


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This dataset cannot give a perfectly balanced view of all clients because history depth differs between clients. Some clients have much less search history than others.

The data also cannot treat all zero values as true zero performance. In particular, availability flags must be checked before interpreting missing or zero-filled data.

The final month should not be used to develop the label logic because it is the natural outcome window. I use March 2026 as the development month and keep the final month sealed for later evaluation.

Another limitation is that this analysis focuses on Search Console performance, so it does not fully explain business outcomes such as conversions or revenue.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the main data limitations described above

# 1. Client history depth
history_check = con.sql(f"""
SELECT
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT CASE
        WHEN report_date <= DATE '2026-03-01' - INTERVAL 365 DAY
        THEN client_hash_id
    END) AS clients_with_12m_history
FROM {fact_daily}
""").df()

print("Client history depth:")
display(history_check)


# 2. GSC-only / unavailable rows
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NOT TRUE
    ) AS gsc_unavailable_rows
FROM {fact_daily}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

print("GSC availability limitation:")
display(availability_check)


# 3. Final month is separated from the development month
window_check = con.sql(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM {fact_daily}
""").df()

print("Available date range:")
display(window_check)

Client history depth:


,clients,clients_with_12m_history
0,55,0


GSC availability limitation:


,total_rows,gsc_available_rows,gsc_unavailable_rows
0,9841378,3611061,6230317


Available date range:


,min_date,max_date,distinct_dates
0,2026-03-01,2026-03-31,31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.